# # Install dependencies

In [1]:
!pip install -q --user scikit-learn torch pytorch-lightning



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


# # Load training data

In [2]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


train  rows= 90

val    rows= 30

test   rows= 30

# # Train (PyTorch Lightning)

In [3]:
"""Define the module in ``src/iris_lightning.py`` so ``infer_lightning.ipynb``
can unpickle it later. A class defined inside this notebook is pickled as
``__main__.IrisLightning`` and won't resolve in another process.
"""
import torch
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
from src.iris_lightning import IrisLightning


In [4]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring;
                    "test" populates `t.X_test / t.y_test`)
    target        — label column name(s); single string or list of strings.
    model_type    — "classifier" / "regressor" / free-form string tag.
    weights_path  — default "src/weights.pkl".

mlflow.pytorch.autolog hooks `LightningModule.save_hyperparameters()` and
`self.log()`. Passing `logger=False` to `pl.Trainer` disables both hooks, so
keep the default Lightning logger active for params/metrics to be captured.
"""
from nubison_model import train

EPOCHS, BATCH = 30, 16

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    X_train = torch.tensor(t.X_train.values, dtype=torch.float32)
    y_train = torch.tensor(t.y_train.values.ravel(), dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True)

    val_loader = None
    if t.X_val is not None and t.y_val is not None:
        X_val = torch.tensor(t.X_val.values, dtype=torch.float32)
        y_val = torch.tensor(t.y_val.values.ravel(), dtype=torch.long)
        val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH)

    model = IrisLightning(lr=0.01)
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        accelerator="cpu",
        enable_progress_bar=False,
        enable_checkpointing=False,
        # Do NOT pass `logger=False` — it disables the mlflow autolog hooks.
    )
    trainer.fit(model, train_loader, val_loader)

    t.save(model)

print(f"run_id: {t.run_id}")


2026/05/18 10:53:12 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/05/18 10:53:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


2026/05/18 10:53:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for pytorch_lightning.


2026/05/18 10:53:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/05/18 10:53:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


...

/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run bouncy-chimp-285 at: http://127.0.0.1:5050/#/experiments/0/runs/1c9dd645d55f4cd487d8dea44d560055


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


run_id: 1c9dd645d55f4cd487d8dea44d560055